In [1]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import scipy as scp
import multiprocessing
import cvxpy as cvp

/Users/andpetrouzeniou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def c_quantile(a, Sigma_Y, k, agg, n_iter, rand):
    
    # k: no. of Y's testing
    # p: length of Y

    N = len(Sigma_Y)
    p = int(N / k)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_Y, n_iter)
    n_draws = -1 * draws
    
    lst = []
    
    for i in range(len(draws)):
        
        draw = draws[i]
        
        sub_lst = []
        for j in range(k):
            
            draw_j = draw[j * p : (j+1) * p]
            Sigma_j = Sigma_Y[j * p : (j+1) * p,]
            Sigma_j = Sigma_j[:,j * p : (j+1) * p]
            
            sub_lst.append(max([abs(sum(draw_j[np.argsort(draw_j)][-agg : ])), abs(sum(-1 * draw_j[np.argsort(-1 * draw_j)][-agg : ]))]))

        lst.append(max(sub_lst))

    return np.quantile(np.array(lst), a)


In [3]:
def d_quantile(b, k, Sigma_X, n_iter, rand):

    N = len(Sigma_X)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_X, n_iter)
    lst = [] # np.amax(draws, axis=1) - np.amax((-1 * draws), axis=1)
    
    for i in range(len(draws)):
        
        draw = draws[i]
        
        comps = []
        
        for row in range(len(draw)):
            for col in range(len(draw)):
                if draw[row] - draw[col] > 0:
                    comps.append(abs(draw[row] - draw[col]) / (Sigma_X[row, row] + Sigma_X[col,col] - 2 * Sigma_X[row,col]) ** 0.5)
        
        lst.append(np.max(np.array(comps)))

    return np.quantile(np.array(lst), b)


In [4]:
def make_I_list(rank_list, X_temp, delta_L, delta_U):
    
    rank = np.max(np.array(rank_list))
    
    # need matrix A[i,j] = X_temp[j] + delta_L[j,i]
    
    X_temp_mat = np.tile(X_temp, (len(X_temp), 1))
    A = X_temp_mat + np.transpose(delta_L)
    
    np.fill_diagonal(A, np.nan)
    A = A[~np.isnan(A)].reshape(A.shape[0], A.shape[1] - 1)

    row_ranks = np.partition(A, -rank, axis=1)[:, -rank]
    
    indices = np.where(X_temp >= row_ranks, 1, 0)
        
    return indices


In [5]:
def iteration_function(l):
    
    Sigma, rank_list, delta_L, delta_U, Sigma_Y, random_ng = l[0], l[1], l[2], l[3], l[4], l[5]
    
    draw = random_ng.multivariate_normal(np.zeros(len(Sigma)), Sigma)

    X_temp = draw.copy()
    rank = np.max(np.array(rank_list))

    I_list = make_I_list(rank_list, X_temp, delta_L, delta_U)

    I_indices = np.where(I_list == 1)[0]  

    return(np.max((abs(X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))


In [6]:
def iteration_function_asymmetric(l):
    
    Sigma, rank_list, delta_L, delta_U, Sigma_Y, random_ng = l[0], l[1], l[2], l[3], l[4], l[5]
    
    draw = random_ng.multivariate_normal(np.zeros(len(Sigma)), Sigma)

    X_temp = draw.copy()
    rank = np.max(np.array(rank_list))

    I_list = make_I_list(rank_list, X_temp, delta_L, delta_U)

    I_indices = np.where(I_list == 1)[0]  

    return(np.max(((X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]), 
              np.min(((X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))
    
    

In [7]:
def c_2_quantile(ab, rank_list, Sigma, k, agg, delta_L, delta_U, n_iter, rand):

    lst = []

    Sigma_Y = Sigma
    
    N = len(Sigma_Y)
    p = int(N / k)
    
    print(len(Sigma_Y))

    for a in range(n_iter):
        print(a, end="")
        # print(i, end="")
        draw = rand.multivariate_normal(np.zeros(len(Sigma)), Sigma)
            
        X_temp = draw.copy()
        X_sel = draw[0:p]

        I_list = make_I_list(rank_list, X_sel, delta_L, delta_U)
        I_indices = np.where(I_list == 1)[0]    
        
        sub_lst = []
        for j in range(k):
            
            draw_j = X_temp[j * p : (j+1) * p]
            draw_j = draw_j[I_indices]
            
            sub_lst.append(max([abs(sum(draw_j[np.argsort(draw_j)][-agg : ])), abs(sum(-1 * draw_j[np.argsort(-1 * draw_j)][-agg : ]))]))

        lst.append(max(sub_lst))        

    return np.quantile(np.array(lst), ab)


In [8]:
def c_2_quantile_asymmetric(ab, rank_list_top, rank_list_bottom, Sigma, delta_L, delta_U, n_iter, rand):

    ab_p = 1 - ab
    ab_p = ab_p / 2
    ab_p = 1 - ab_p
    
    lst_l = []
    lst_r = []

    Sigma_Y = Sigma

    for a in range(n_iter):
        print(a, end="")
        # print(i, end="")
        draw = rand.multivariate_normal(np.zeros(len(Sigma)), Sigma)
            
        X_temp = draw.copy()
        
        I_list_top = make_I_list(rank_list_top, X_temp, delta_L, delta_U)
        I_list_bot = make_I_list(rank_list_bottom, (-1 * X_temp), np.transpose(delta_L), np.transpose(delta_U))
        
        print()
        print(len(I_list_top))
        print(len(I_list_bot))
        
        I_indices_top = np.where(I_list_top == 1)[0]    
        I_indices_bot = np.where(I_list_bot == 1)[0]  

        tops = X_temp[I_indices_top]
        bots = X_temp[I_indices_bot]
        
        tops2 = np.tile(tops, (len(bots), 1)) # len(bots) length, len(tops) width
        bots2 = np.transpose(np.tile(bots, (len(tops), 1)))
        
        err = tops2-bots2
        
        Sigma_tops = np.diag(Sigma_Y)[I_indices_top]
        Sigma_bots = np.diag(Sigma_Y)[I_indices_bot]
        
        Sigma_tops2 = np.tile(Sigma_tops, (len(bots), 1)) # len(bots) length, len(tops) width
        Sigma_bots2 = np.transpose(np.tile(Sigma_bots, (len(tops), 1)))
        
        err = err / (Sigma_tops2 + Sigma_bots2) ** 0.5  
        
        lst_r.append(np.max(err))
        lst_l.append(np.min(err))

    return([np.quantile(-1 * np.array(lst_l), ab_p), np.quantile(np.array(lst_r), ab_p)])


In [9]:
def c_3_quantile(ab, subset, Sigma, n_iter, rand):
    
    lst = []
    N = len(Sigma)
    
    for i in range(n_iter):
        draw = rand.multivariate_normal(np.zeros(N), Sigma)
        
        draw = draw[subset]
        diag = np.diagonal(Sigma)[subset]
        lst.append(np.max(abs(draw / (diag ** (1/2)))))
            
    return np.quantile(np.array(lst), ab)


In [10]:
# Andrews et al. projection CI

def CI_proj(rank_list, y_vec_list, a, Sigma_Y, k, agg, n_iter, quantile_dict, rand):

    th_hat_lst = [] 
    y_vec = y_vec_list[0]
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])
        
    print(Sigma_Y)

    c_a = c_quantile(a, Sigma_Y, k, agg, n_iter, rand = rand)
        
    print(c_a)
        
    CI_list = []
    
    for i in range(k):
        y_vec_i = y_vec_list[i]
        center = sum([y_vec_i[j] for j in th_hat_lst])
            
        CI_t = [center - c_a, center + c_a]
        CI_list.append(CI_t)

    return CI_list


In [11]:
def two_step_CI(rank_list, y_vec_list, a, b, Sigma, k, agg, n_iter, quantile_dict, rand):
    
    Sigma_Y = Sigma
    y_vec = y_vec_list[0]
    
    p = int(len(Sigma) / k)

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])
        
    Sigma_c = Sigma_Y[:p,:]
    Sigma_c = Sigma_c[:,:p]

    d = d_quantile(1 - b, k, Sigma_c, n_iter, rand = rand)
    print()
    
    l = np.tile(y_vec, (len(y_vec), 1))
    r = np.transpose(l)
    
    print(l)
    print(np.shape(Sigma_c))

    var_mat = np.tile(np.diag(Sigma_c), (len(np.diag(Sigma_c)), 1))
    var_mat += np.transpose(var_mat)
    var_mat -= 2 * Sigma_c
    var_mat = var_mat ** 0.5
    
    L = r - l - d * var_mat
    U = r - l + d * var_mat
    
    # L[1,2] = r[1,2] - l[1,2] - d = l[2,1] - l[1,2] - d = y_vec[1] - y_vec[2] - d
    
    bd = c_2_quantile(1-a+b, rank_list, Sigma_Y, k, agg, L, U, int(n_iter), rand = rand)
    print()
    print(bd)
    
    CI_list = []
    
    for i in range(k):
        y_vec_i = y_vec_list[i]
        center = sum([y_vec_i[j] for j in th_hat_lst])
            
        CI_t = [center - bd, center + bd]
        CI_list.append(CI_t)

    return CI_list


In [12]:
def two_step_CI_asymmetric(rank_list_top, rank_list_bot, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):
    
    Sigma_Y = Sigma
    k = len(y_vec)

    th_hat_lst_top = [] 
    
    for i in range(len(rank_list_top)):
        th_hat_lst_top.append(np.argsort(y_vec)[len(y_vec)-rank_list_top[i]])
        
    th_hat_lst_bot = [] 
    
    for i in range(len(rank_list_bot)):
        th_hat_lst_bot.append(np.argsort(-1 * y_vec)[len(y_vec)-rank_list_bot[i]])

    d = d_quantile(1 - b / 2, k, Sigma_Y, n_iter, rand = rand)
    print()
    
    l = np.tile(y_vec, (len(y_vec), 1))
    r = np.transpose(l)
    
    var_mat = np.tile(np.diag(Sigma_Y), (len(np.diag(Sigma_Y)), 1))
    var_mat += np.transpose(var_mat)
    var_mat -= 2 * Sigma_Y
    var_mat = var_mat ** 0.5
    
    L = r - l - d * var_mat
    U = r - l + d * var_mat
    
    # L[1,2] = r[1,2] - l[1,2] - d = l[2,1] - l[1,2] - d = y_vec[1] - y_vec[2] - d
    
    bd = c_2_quantile_asymmetric(1 - a + b, rank_list_top, rank_list_bot, Sigma, L, U, int(n_iter), rand = rand)
    bd_l = bd[0]
    bd_r = bd[1]
    print()
    
    CI_list = []
    
    for i in range(len(rank_list_top)):
        for j in range(len(rank_list_bot)):
            th_hat_top = th_hat_lst_top[i]
            th_hat_bot = th_hat_lst_bot[j]
            
            CI_t = [(y_vec[th_hat_top] - y_vec[th_hat_bot]) - bd_r * ((Sigma_Y[th_hat_top, th_hat_top] + Sigma_Y[th_hat_bot, th_hat_bot]) ** 0.5), (y_vec[th_hat_top] - y_vec[th_hat_bot]) + bd_l * ((Sigma_Y[th_hat_top, th_hat_top] + Sigma_Y[th_hat_bot, th_hat_bot]) ** 0.5)]
            CI_list.append(CI_t)

    return CI_list


In [13]:
Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_Seattle_urban_extended.csv",index_col = 0).to_numpy()
X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_Seattle_urban_extended.csv",index_col = 0).to_numpy().flatten()

p = int(len(X) / 5)

In [14]:
X_lst = []
for i in range(5):
    X_lst.append(X[(i * p) : ((i + 1) * p)])

In [15]:
rng = np.random.default_rng(42)

top_pairs = int(p / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([i for i in range(1,top_pairs)], X_lst, 0.95, Sigma, 5, int(p / 5), 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([i for i in range(1,top_pairs)], X_lst, 0.05, 0.005, Sigma, 5, int(p / 5), 1000, {}, rng))

# print("Two Step")
# two_step_a = pd.DataFrame(two_step_CI_asymmetric([i for i in range(1,top_pairs)], [i for i in range(1,top_pairs)], X, 0.05, 0.005, Sigma, 10000, {}, rng))



Projection
[[ 3.73949681e-04 -2.91991238e-07  1.73856176e-06 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-2.91991238e-07  3.32986896e-04  5.21899137e-06 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 1.73856176e-06  5.21899137e-06  8.28766266e-04 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  4.16963983e-04
   2.37143145e-05  2.37143145e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  2.37143145e-05
   1.19916521e-03  2.70710349e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  2.37143145e-05
   2.70710349e-05  3.18857243e-03]]
2.233020599814623
Two Step

[[ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 ...
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.073290

In [16]:
print(two_step)
print(proj)

          0         1
0  0.243345  4.657474
1 -8.235509 -3.821380
2 -0.288118  4.126011
3  0.118655  4.532784
4 -0.636020  3.778109
          0         1
0  0.217389  4.683430
1 -8.261465 -3.795424
2 -0.314074  4.151967
3  0.092699  4.558740
4 -0.661976  3.804065


In [17]:
X_lst = []
for i in range(3):
    X_lst.append(X[(i * p) : ((i + 1) * p)])

In [18]:
rng = np.random.default_rng(42)

Sigma_2 = Sigma[:(3*p),:(3*p)]

top_pairs = int(p / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([i for i in range(1,top_pairs)], X_lst, 0.95, Sigma_2, 3, int(p / 5), 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([i for i in range(1,top_pairs)], X_lst, 0.05, 0.005, Sigma_2, 3, int(p / 5), 1000, {}, rng))

# print("Two Step")
# two_step_a = pd.DataFrame(two_step_CI_asymmetric([i for i in range(1,top_pairs)], [i for i in range(1,top_pairs)], X, 0.05, 0.005, Sigma, 10000, {}, rng))


Projection
[[ 3.73949681e-04 -2.91991238e-07  1.73856176e-06 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-2.91991238e-07  3.32986896e-04  5.21899137e-06 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 1.73856176e-06  5.21899137e-06  8.28766266e-04 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  1.41546719e-03
   3.46649478e-05  3.46649478e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  3.46649478e-05
   1.30816976e-03  4.64512694e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  3.46649478e-05
   4.64512694e-05  2.89319908e-03]]
2.2001383563247705
Two Step

[[ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 ...
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329

In [19]:
print(two_step)
print(proj)

          0         1
0  0.270592  4.630227
1 -8.208262 -3.848627
2 -0.260871  4.098764
          0         1
0  0.250271  4.650548
1 -8.228583 -3.828306
2 -0.281192  4.119084


In [20]:
X_lst = [X[0:p], X[(3*p):(4*p)], X[(4*p):]]

In [21]:
rng = np.random.default_rng(42)

indices = [i for i in range(p)]
indices2 = [i for i in range((3*p),(5*p))]

indices = np.concatenate((indices, indices2))

Sigma_3 = Sigma[indices,:]
Sigma_3 = Sigma_3[:,indices]

top_pairs = int(p / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([i for i in range(1,top_pairs)], X_lst, 0.95, Sigma_3, 3, int(p / 5), 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([i for i in range(1,top_pairs)], X_lst, 0.05, 0.005, Sigma_3, 3, int(p / 5), 1000, {}, rng))

# print("Two Step")
# two_step_a = pd.DataFrame(two_step_CI_asymmetric([i for i in range(1,top_pairs)], [i for i in range(1,top_pairs)], X, 0.05, 0.005, Sigma, 10000, {}, rng))


Projection
[[ 3.73949681e-04 -2.91991238e-07  1.73856176e-06 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-2.91991238e-07  3.32986896e-04  5.21899137e-06 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 1.73856176e-06  5.21899137e-06  8.28766266e-04 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  4.16963983e-04
   2.37143145e-05  2.37143145e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  2.37143145e-05
   1.19916521e-03  2.70710349e-05]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  2.37143145e-05
   2.70710349e-05  3.18857243e-03]]
2.138372593772924
Two Step

[[ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.07329087]
 ...
 [ 0.06958566  0.0442786   0.03276513 ... -0.01241239 -0.06300419
   0.073290

In [22]:
print(two_step)
print(proj)

          0         1
0  0.298587  4.602232
1  0.173897  4.477541
2 -0.580778  3.722867
          0         1
0  0.312037  4.588782
1  0.187347  4.464092
2 -0.567328  3.709417


In [23]:
two_step = np.array(two_step)
left = two_step[:,0]

print(len(left[left < 0]) / len(left))

two_step

0.3333333333333333


array([[ 0.29858721,  4.60223191],
       [ 0.17389679,  4.47754149],
       [-0.5807778 ,  3.7228669 ]])

In [24]:
proj = np.array(proj)
left = proj[:,0]

print(len(left[left < 0]) / len(left))

proj

0.3333333333333333


array([[ 0.31203697,  4.58878216],
       [ 0.18734655,  4.46409173],
       [-0.56732805,  3.70941714]])

In [25]:
rng = np.random.default_rng(42)

top_pairs = int(len(X) / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([i for i in range(1,top_pairs)], [1], X, 0.95, Sigma, 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([i for i in range(1,top_pairs)], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))

print("Two Step")
two_step_a = pd.DataFrame(two_step_CI_asymmetric([i for i in range(1,top_pairs)], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))



Projection


TypeError: CI_proj() missing 1 required positional argument: 'rand'

In [ ]:
two_step = np.array(two_step)
left = two_step[:,0]

len(left[left < 0]) / len(left)

two_step

In [ ]:
proj = np.array(proj)
left = proj[:,0]

print(len(left[left < 0]) / len(left))

proj

In [ ]:
rng = np.random.default_rng(42)

top_pairs = int(len(X) / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([1], [1], X, 0.95, Sigma, 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([1], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))

print("Two Step")
two_step_a = pd.DataFrame(two_step_CI_asymmetric([1], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))



In [ ]:
two_step

In [ ]:
proj

In [ ]:
Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy()
X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy().flatten()
Y = np.array(pd.read_csv("Input_Data_Processed_Urban/Y_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy()).flatten()


In [ ]:
rng = np.random.default_rng(42)

top_pairs = int(len(X) / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([i for i in range(1,top_pairs)], [i for i in range(1,top_pairs)], X, 0.95, Sigma, 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([i for i in range(1,top_pairs)], [i for i in range(1,top_pairs)], X, 0.05, 0.005, Sigma, 1000, {}, rng))

print("Two Step A")
two_step_a = pd.DataFrame(two_step_CI_asymmetric([i for i in range(1,top_pairs)], [i for i in range(1,top_pairs)], X, 0.05, 0.005, Sigma, 1000, {}, rng))



In [ ]:
two_step = np.array(two_step)
left = two_step[:,0]

print(len(left[left < 0]) / len(left))

two_step

In [ ]:
proj = np.array(proj)
left = proj[:,0]

print(len(left[left < 0]) / len(left))

proj

In [ ]:
rng = np.random.default_rng(42)

top_pairs = int(len(X) / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([i for i in range(1,top_pairs)], [1], X, 0.95, Sigma, 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([i for i in range(1,top_pairs)], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))

print("Two Step")
two_step_a = pd.DataFrame(two_step_CI_asymmetric([i for i in range(1,top_pairs)], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))



In [ ]:
two_step = np.array(two_step)
left = two_step[:,0]

len(left[left < 0]) / len(left)

two_step

In [ ]:
proj = np.array(proj)
left = proj[:,0]

print(len(left[left < 0]) / len(left))

proj

In [ ]:
rng = np.random.default_rng(42)

top_pairs = int(len(X) / 5)

print("Projection")
proj = pd.DataFrame(CI_proj([1], [1], X, 0.95, Sigma, 1000, {}, rng))

print("Two Step")
two_step = pd.DataFrame(two_step_CI([1], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))

print("Two Step")
two_step_a = pd.DataFrame(two_step_CI_asymmetric([1], [1], X, 0.05, 0.005, Sigma, 1000, {}, rng))



In [ ]:
two_step

In [ ]:
proj